# 03 — Model Training Orchestrator
## Hackathon IBM — Détection de Fraude Bancaire

**Objectif unique** : entraîner et évaluer systématiquement **3 modèles** (CatBoost, XGBoost, TabNet) sur les **10 datasets d'entraînement** à différents ratios d'undersampling, avec évaluation sur un **set de test unique et fixe**.

### Matériel cible
Machine à **4× NVIDIA T4** → les 3 modèles sont configurés pour exploiter le GPU, avec **auto-fallback CPU** si les libs CUDA ne sont pas détectées (utile pour tester localement).

### Livrables produits
- `logs_training/experiment_results.csv` — tableau global à 13 colonnes, mis à jour à chaque itération.
- `logs_training/models/<algo>_train_<ratio>.{cbm|ubj|zip}` — poids natifs de chaque modèle entraîné.
- Un **leaderboard final** en fin de notebook avec le meilleur couple (modèle, ratio) selon PR-AUC et F1.

### Métriques suivies (classe positive = fraude)
- **F1** (métrique reine)
- **Recall** (priorité fraude : ne pas rater)
- **Precision** (coût des faux positifs)
- **PR-AUC** (Average Precision — cruciale en déséquilibre extrême)
- **ROC-AUC**, **Accuracy**
- **Matrice de confusion** complète : TN / FP / FN / TP

---
## 0. Installation (à exécuter une seule fois sur la machine T4)

Décommente la cellule ci-dessous si les libs ne sont pas déjà installées.

In [ ]:
# !pip install -q catboost xgboost pytorch-tabnet
# !pip install -q torch --index-url https://download.pytorch.org/whl/cu121   # CUDA 12.1 build
# !pip install -q scikit-learn pandas numpy joblib tqdm pyarrow

---
## 1. Imports & détection du matériel

In [ ]:
import os
import re
import gc
import time
import glob
import json
import warnings
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score, recall_score, precision_score, accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import catboost as cb
import xgboost as xgb

import torch
from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

# --- Détection du matériel ---
CUDA_AVAILABLE = torch.cuda.is_available()
N_GPUS = torch.cuda.device_count() if CUDA_AVAILABLE else 0
DEVICE = 'cuda' if CUDA_AVAILABLE else 'cpu'
XGB_VERSION = tuple(int(x) for x in xgb.__version__.split('.')[:2])

print('=== Environment ===')
print(f'pandas   : {pd.__version__}')
print(f'numpy    : {np.__version__}')
print(f'torch    : {torch.__version__}')
print(f'catboost : {cb.__version__}')
print(f'xgboost  : {xgb.__version__}')
print(f'CUDA     : {CUDA_AVAILABLE} (devices={N_GPUS})')
if CUDA_AVAILABLE:
    for i in range(N_GPUS):
        print(f'  GPU[{i}] : {torch.cuda.get_device_name(i)}')
if not CUDA_AVAILABLE:
    print('\n⚠ Aucune CUDA détectée → fallback CPU. Les temps seront plus longs.')

---
## 2. Configuration globale

In [ ]:
DATA_DIR    = Path('.')
TRAIN_GLOB  = 'prepared_train_*.parquet'
TEST_FILE   = 'prepared_test_050.0_pct.parquet'

LOG_DIR     = Path('logs_training')
MODELS_DIR  = LOG_DIR / 'models'
RESULTS_CSV = LOG_DIR / 'experiment_results.csv'

LOG_DIR.mkdir(exist_ok=True, parents=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)

TARGET = 'is_fraud'
RANDOM_STATE = 42

# Budget entraînement (volontairement modéré pour itérer vite)
CATBOOST_ITER = 2000
XGBOOST_ITER  = 1500
TABNET_EPOCHS = 80

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f'Logs        -> {LOG_DIR.resolve()}')
print(f'Models      -> {MODELS_DIR.resolve()}')
print(f'Results CSV -> {RESULTS_CSV.resolve()}')

---
## 3. Helpers génériques

In [ ]:
_RATIO_RE = re.compile(r'(\d+\.\d+)')


def extract_ratio(path) -> float:
    """'prepared_train_010.0_pct.parquet' -> 10.0"""
    m = _RATIO_RE.search(Path(path).stem)
    return float(m.group(1)) if m else np.nan


def load_dataset(path) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    """Charge un .parquet préparé. Retourne (X, y, cat_feature_names)."""
    df = pd.read_parquet(path)
    y = df[TARGET].astype(np.int8)
    X = df.drop(columns=[TARGET])
    cat_features = [c for c, dt in X.dtypes.items() if str(dt) == 'category']
    return X, y, cat_features


def align_categories(X_train: pd.DataFrame, X_test: pd.DataFrame,
                      cat_features: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Force les mêmes catégories dans train et test (union des modalités).
    Indispensable pour XGBoost (enable_categorical=True) qui exige la cohérence."""
    X_train = X_train.copy()
    X_test  = X_test.copy()
    for c in cat_features:
        all_cats = (
            pd.api.types.union_categoricals([X_train[c], X_test[c]], sort_categories=True)
              .categories
        )
        X_train[c] = pd.Categorical(X_train[c], categories=all_cats)
        X_test[c]  = pd.Categorical(X_test[c],  categories=all_cats)
    return X_train, X_test


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray) -> Dict[str, float]:
    """Calcule toutes les métriques de fraude + matrice de confusion en un appel."""
    # Gère les cas dégénérés (aucun positif prédit, ou aucune fraude dans le test)
    metrics = {
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Accuracy':  accuracy_score(y_true, y_pred),
        'PR-AUC':    average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        'ROC-AUC':   roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
    }
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0
    metrics.update({'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)})
    return metrics


def append_result_row(row: dict, csv_path: Path) -> None:
    """Append thread-safe d'une ligne au CSV global. Crée le header si le fichier n'existe pas."""
    df_row = pd.DataFrame([row])
    header = not csv_path.exists()
    df_row.to_csv(csv_path, mode='a', header=header, index=False)

---
## 4. Entraîneur — CatBoost (GPU natif, gère les catégorielles sans encoding)

**Pourquoi CatBoost est particulièrement adapté** : supporte nativement les colonnes `category` via `cat_features`, robuste au déséquilibre (`auto_class_weights='Balanced'`), GPU via `task_type='GPU'`. **Zéro preprocessing** nécessaire.

In [ ]:
def train_catboost(X_train, y_train, X_test, y_test, cat_features) -> Tuple:
    """Entraîne CatBoost, retourne (model, y_pred, y_proba, training_time_s)."""

    # CatBoost attend les catégorielles en str (pas de dtype 'category' directement)
    Xtr = X_train.copy()
    Xte = X_test.copy()
    for c in cat_features:
        Xtr[c] = Xtr[c].astype(str).fillna('missing')
        Xte[c] = Xte[c].astype(str).fillna('missing')

    params = dict(
        iterations=CATBOOST_ITER,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE,
        eval_metric='PRAUC',
        loss_function='Logloss',
        auto_class_weights='Balanced',   # gère le déséquilibre automatiquement
        od_type='Iter',
        od_wait=100,
        verbose=False,
        allow_writing_files=False,
    )
    if CUDA_AVAILABLE:
        params.update(dict(task_type='GPU', devices='0'))
    else:
        params.update(dict(task_type='CPU', thread_count=-1))

    model = cb.CatBoostClassifier(**params)

    t0 = time.perf_counter()
    model.fit(
        Xtr, y_train,
        cat_features=cat_features,
        eval_set=(Xte, y_test),
        use_best_model=True,
    )
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(Xte)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_catboost(model, path: Path) -> None:
    model.save_model(str(path.with_suffix('.cbm')))

---
## 5. Entraîneur — XGBoost (GPU, catégorielles Pandas activées)

**Compatibilité versions** :
- XGBoost ≥ 2.0 : `tree_method='hist'` + `device='cuda'`
- XGBoost 1.x  : `tree_method='gpu_hist'`

**Catégorielles** : `enable_categorical=True` depuis 1.5+. On **unifie** les catégories train/test en amont pour éviter l'erreur "unknown category".

In [ ]:
def train_xgboost(X_train, y_train, X_test, y_test, cat_features) -> Tuple:
    """Entraîne XGBoost avec categorical natif + GPU."""

    # Alignement des catégories train <-> test (requis par XGBoost)
    X_train_al, X_test_al = align_categories(X_train, X_test, cat_features)

    params = dict(
        n_estimators=XGBOOST_ITER,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=1.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        reg_alpha=0.0,
        objective='binary:logistic',
        eval_metric='aucpr',       # Area Under PR curve — adapté au déséquilibre
        scale_pos_weight=float((y_train == 0).sum() / max((y_train == 1).sum(), 1)),
        random_state=RANDOM_STATE,
        enable_categorical=True,
        n_jobs=-1,
        early_stopping_rounds=100,
        verbosity=0,
    )
    if CUDA_AVAILABLE:
        if XGB_VERSION >= (2, 0):
            params.update(dict(tree_method='hist', device='cuda'))
        else:
            params.update(dict(tree_method='gpu_hist', predictor='gpu_predictor'))
    else:
        params.update(dict(tree_method='hist'))

    model = xgb.XGBClassifier(**params)

    t0 = time.perf_counter()
    model.fit(
        X_train_al, y_train,
        eval_set=[(X_test_al, y_test)],
        verbose=False,
    )
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(X_test_al)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_xgboost(model, path: Path) -> None:
    # .ubj = UBJSON natif XGBoost (recommandé), portable et robuste aux versions
    model.save_model(str(path.with_suffix('.ubj')))

---
## 6. Entraîneur — TabNet (CUDA, preprocessing maison)

**TabNet n'accepte que du numérique dense**. On construit un petit pipeline :
- Catégorielles → LabelEncoder fitted sur l'union train+test (évite `unknown category`). Les indices et cardinalités sont passés via `cat_idxs` / `cat_dims`.
- Numériques → imputation médiane (pas de NaN accepté) + StandardScaler (TabNet est un réseau de neurones).
- Device `cuda` si dispo, sinon `cpu`.

In [ ]:
def build_tabnet_preprocessing(X_train: pd.DataFrame, X_test: pd.DataFrame,
                                cat_features: List[str]) -> dict:
    """Construit le preprocessing TabNet et retourne un dict avec tout le nécessaire.

    Retour :
      - X_train_np, X_test_np : np.ndarray float32 prêts pour TabNet
      - cat_idxs  : indices des colonnes catégorielles après encoding
      - cat_dims  : cardinalité (nb de modalités distinctes) par colonne cat
    """
    num_cols = [c for c in X_train.columns if c not in cat_features]

    # Ordre des colonnes stable : cat d'abord, puis num (plus simple pour cat_idxs)
    ordered_cols = cat_features + num_cols
    X_tr = X_train[ordered_cols].copy()
    X_te = X_test[ordered_cols].copy()

    # --- Encoders catégoriels fitted sur l'union pour éviter les unseen ---
    cat_dims = []
    for c in cat_features:
        enc = LabelEncoder()
        combined = pd.concat([X_tr[c].astype(str), X_te[c].astype(str)], axis=0)
        enc.fit(combined.fillna('missing'))
        X_tr[c] = enc.transform(X_tr[c].astype(str).fillna('missing'))
        X_te[c] = enc.transform(X_te[c].astype(str).fillna('missing'))
        cat_dims.append(len(enc.classes_))

    cat_idxs = list(range(len(cat_features)))  # les N premières colonnes

    # --- Numériques : imputation + scaling ---
    imputer = SimpleImputer(strategy='median')
    scaler  = StandardScaler()

    X_tr[num_cols] = imputer.fit_transform(X_tr[num_cols])
    X_te[num_cols] = imputer.transform(X_te[num_cols])
    X_tr[num_cols] = scaler.fit_transform(X_tr[num_cols])
    X_te[num_cols] = scaler.transform(X_te[num_cols])

    return {
        'X_train_np': X_tr.values.astype(np.float32),
        'X_test_np':  X_te.values.astype(np.float32),
        'cat_idxs':   cat_idxs,
        'cat_dims':   cat_dims,
    }


def train_tabnet(X_train, y_train, X_test, y_test, cat_features) -> Tuple:
    """Entraîne TabNet sur GPU si dispo. Minibatchs petits sur petits datasets."""

    prep = build_tabnet_preprocessing(X_train, X_test, cat_features)

    # Tailles adaptatives : sur un dataset de 500 lignes, batch=256 est déjà bien
    n = len(X_train)
    batch_size        = min(1024, max(32, 2 ** int(np.log2(max(n // 8, 32)))))
    virtual_batch_size = min(128, max(16, batch_size // 8))

    model = TabNetClassifier(
        cat_idxs=prep['cat_idxs'],
        cat_dims=prep['cat_dims'],
        cat_emb_dim=4,
        n_d=16, n_a=16, n_steps=3,
        gamma=1.3, lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params=dict(step_size=20, gamma=0.9),
        seed=RANDOM_STATE,
        device_name=DEVICE,
        verbose=0,
    )

    t0 = time.perf_counter()
    model.fit(
        X_train=prep['X_train_np'], y_train=y_train.values.astype(np.int64),
        eval_set=[(prep['X_test_np'], y_test.values.astype(np.int64))],
        eval_name=['test'],
        eval_metric=['auc'],
        max_epochs=TABNET_EPOCHS,
        patience=20,
        batch_size=batch_size,
        virtual_batch_size=virtual_batch_size,
        num_workers=0,
        drop_last=False,
        weights=1,          # 1 = class weighting automatique (anti-déséquilibre)
    )
    elapsed = time.perf_counter() - t0

    y_proba = model.predict_proba(prep['X_test_np'])[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba, elapsed


def save_tabnet(model, path: Path) -> None:
    # pytorch-tabnet sauvegarde en .zip (architecture + poids + encoders internes)
    model.save_model(str(path))   # ajoute .zip automatiquement

---
## 7. Registre des modèles

In [ ]:
MODEL_REGISTRY = {
    'CatBoost': dict(train_fn=train_catboost, save_fn=save_catboost),
    'XGBoost':  dict(train_fn=train_xgboost,  save_fn=save_xgboost),
    'TabNet':   dict(train_fn=train_tabnet,   save_fn=save_tabnet),
}

print('Modèles enregistrés :', list(MODEL_REGISTRY.keys()))

---
## 8. Chargement unique du test set

In [ ]:
test_path = DATA_DIR / TEST_FILE
X_test, y_test, cat_features_test = load_dataset(test_path)

print(f'Test set  : {test_path.name}')
print(f'  shape   : {X_test.shape}')
print(f'  fraude  : {y_test.mean()*100:.3f} %  ({int(y_test.sum())} positifs / {len(y_test)})')
print(f'  cat cols: {len(cat_features_test)}  -> {cat_features_test}')
print(f'  mémoire : {X_test.memory_usage(deep=True).sum()/1024**2:.2f} MB')

---
## 9. Boucle d'expérimentation principale

In [ ]:
def run_single_experiment(train_path: Path, model_name: str, cfg: dict,
                           X_test, y_test) -> dict:
    """Entraîne UN modèle sur UN dataset, évalue, sauvegarde poids + ligne CSV.
    Retourne le dict de résultats."""
    ratio = extract_ratio(train_path)

    X_train, y_train, cat_features = load_dataset(train_path)

    print(f'  ├─ {model_name:<8} | train={len(X_train):>6,} | '
          f'fraud_train={y_train.mean()*100:>5.2f}% ', end='', flush=True)

    t_global = time.perf_counter()
    model, y_pred, y_proba, train_time = cfg['train_fn'](
        X_train, y_train, X_test, y_test, cat_features
    )
    metrics = compute_metrics(y_test.values, y_pred, y_proba)

    # Sauvegarde des poids
    ratio_tag  = f'{ratio:05.1f}'.replace('.', '_')
    model_path = MODELS_DIR / f'{model_name.lower()}_train_{ratio_tag}'
    cfg['save_fn'](model, model_path)

    row = {
        'Model':                 model_name,
        'Dataset_Ratio':         ratio,
        'Dataset_File':          Path(train_path).name,
        'N_Train':               len(X_train),
        'Fraud_Rate_Train_%':    round(y_train.mean() * 100, 4),
        **{k: round(v, 6) if isinstance(v, float) else v for k, v in metrics.items()},
        'Training_Time_Seconds': round(train_time, 2),
        'Total_Time_Seconds':    round(time.perf_counter() - t_global, 2),
    }
    append_result_row(row, RESULTS_CSV)

    print(f'| F1={metrics["F1"]:.3f} | PR-AUC={metrics["PR-AUC"]:.3f} | '
          f'Recall={metrics["Recall"]:.3f} | time={train_time:6.1f}s')

    # Libération mémoire : le modèle est déjà sauvegardé sur disque
    del model, X_train, y_train, y_pred, y_proba
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()

    return row

In [ ]:
# --- Si on relance le notebook, on repart d'un CSV propre (commente pour accumuler) ---
if RESULTS_CSV.exists():
    RESULTS_CSV.unlink()

train_files = sorted(DATA_DIR.glob(TRAIN_GLOB))
print(f'{len(train_files)} datasets d\'entraînement à traiter :')
for f in train_files:
    print(f'  - {f.name:<40} (ratio={extract_ratio(f):.1f}%)')
print()

t_all = time.perf_counter()
all_results = []

for i, train_path in enumerate(train_files, 1):
    ratio = extract_ratio(train_path)
    print(f'[{i}/{len(train_files)}] {train_path.name}  (ratio={ratio:.1f}%)')

    for model_name, cfg in MODEL_REGISTRY.items():
        try:
            row = run_single_experiment(train_path, model_name, cfg, X_test, y_test)
            all_results.append(row)
        except Exception as e:
            print(f'  ├─ {model_name:<8} | ❌ ERREUR : {type(e).__name__}: {e}')
            append_result_row({
                'Model': model_name,
                'Dataset_Ratio': ratio,
                'Dataset_File': train_path.name,
                'Error': f'{type(e).__name__}: {e}',
            }, RESULTS_CSV)
    print()

print(f'=== TERMINÉ en {(time.perf_counter()-t_all)/60:.1f} min ===')
print(f'Résultats sauvegardés dans : {RESULTS_CSV}')

---
## 10. Analyse des résultats & leaderboard

In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f'Résultats chargés : {results.shape}')
results

In [ ]:
# --- Leaderboard : top 10 couples (Model, Ratio) par PR-AUC ---
if 'PR-AUC' in results.columns:
    lb = (
        results
        .dropna(subset=['PR-AUC'])
        .sort_values('PR-AUC', ascending=False)
        .loc[:, ['Model', 'Dataset_Ratio', 'F1', 'Recall', 'Precision',
                 'PR-AUC', 'ROC-AUC', 'TP', 'FP', 'FN', 'TN',
                 'Training_Time_Seconds']]
        .reset_index(drop=True)
    )
    print('=== TOP 10 par PR-AUC ===')
    print(lb.head(10).to_string(index=False))

    print('\n=== TOP 10 par F1 ===')
    print(lb.sort_values('F1', ascending=False).head(10).to_string(index=False))

    print('\n=== Meilleur résultat par modèle ===')
    best_per_model = lb.loc[lb.groupby('Model')['PR-AUC'].idxmax()].sort_values('PR-AUC', ascending=False)
    print(best_per_model.to_string(index=False))

In [ ]:
# --- Visualisation : comparaison des modèles à travers les ratios ---
import matplotlib.pyplot as plt

metrics_to_plot = ['F1', 'Recall', 'Precision', 'PR-AUC']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, m in zip(axes.ravel(), metrics_to_plot):
    for model_name in results['Model'].dropna().unique():
        sub = (results[results['Model'] == model_name]
               .dropna(subset=[m])
               .sort_values('Dataset_Ratio'))
        ax.plot(sub['Dataset_Ratio'], sub[m], marker='o', lw=2, label=model_name)
    ax.set_xlabel('Ratio undersampling train (% de fraude)')
    ax.set_ylabel(m)
    ax.set_title(f'{m} vs ratio d\'undersampling')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(LOG_DIR / 'metrics_vs_ratio.png', dpi=110, bbox_inches='tight')
plt.show()
print(f'Graphique sauvegardé : {LOG_DIR / "metrics_vs_ratio.png"}')

---
## 11. Bilan

**Artefacts générés** :
- `logs_training/experiment_results.csv` — 30 lignes (10 ratios × 3 modèles).
- `logs_training/models/*.{cbm,ubj,zip}` — 30 poids de modèles rechargeables.
- `logs_training/metrics_vs_ratio.png` — visualisation comparative.

**Prochaines étapes possibles** :
1. **Tuning d'hyperparamètres** sur le meilleur couple (modèle, ratio) via Optuna.
2. **Optimisation du seuil de décision** (pour l'instant fixé à 0.5) : maximiser F1 ou une cost-function métier.
3. **Stacking / blending** des 3 modèles si leurs erreurs sont décorrélées.
4. **SHAP** sur le meilleur modèle pour vérifier que les features dominantes sont bien les features généralisables (velocity, ratios, is_out_of_state, MCC) et non des artefacts.